# AIC211 Project (Topic 10): Threat Intelligence / ATT&CK Tactic Classification

Multilabel classification of cyber threat intelligence sentences into the 15 MITRE ATT&CK Enterprise tactics (v20.0).

**Files needed in `./data/`:** `enterprise-attack-20.0.json` (or `20_0.json`).
Optional: `val_probs_bert.npy`, `val_predictions_bert.csv`, `test_predictions.csv` (outputs of an earlier DistilBERT run, used when `torch` is not available) and `models/bert/` (a saved model).

**Run order:** run all cells from top to bottom. The test set is used once, in Step 8.

In [1]:
"""
AIC211 PROJECT (Topic 10): Threat Intelligence / ATT&CK Classification Pipeline
===============================================================================
Task:    Sentence-level MULTILABEL classification of cyber threat intelligence (CTI)
         text into the 15 MITRE ATT&CK Enterprise tactics (ATT&CK v20.0).

Dataset: MITRE ATT&CK Enterprise STIX 2.1 bundle ("procedure examples")
  https://github.com/mitre-attack/attack-stix-data
  Put the file in:  ./data/enterprise-attack-20.0.json
  (the name enterprise-attack-20_0.json also works)

This notebook builds a complete, leakage-aware pipeline:
  0. Load the raw STIX data
  1. Build the (sentence, technique, tactic) table
  2. Preprocessing: remove IDs, mask actor names and IOCs, merge duplicates
  3. Group-aware, multilabel-stratified train / validation / test split
  4. Baseline models (TF-IDF + Logistic Regression / Linear SVM)
  5. Improved model (fine-tuned DistilBERT: live if torch is available, otherwise saved outputs)
  6. Per-tactic threshold tuning with an honest 2-fold grouped CV
  7. Model comparison on the validation set
  8. FINAL evaluation on the test set (used once) + bootstrap confidence intervals
  9. Error analysis
 10. Security impact analysis
 11. Interpretability (word importance)
 12. Robustness to mild input noise
 13. Visualizations
 14. Optional web demo
"""

import os
import re
import sys
import json
import glob
import time
import random
import warnings
import textwrap
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SCRIPT_DIR = os.getcwd()
DATA_DIR = os.path.join(SCRIPT_DIR, "data")
MODEL_DIR = os.path.join(SCRIPT_DIR, "models", "bert")
os.makedirs(DATA_DIR, exist_ok=True)

# Improved model (DistilBERT) settings
RUN_BERT = True                  # set False to skip the transformer completely
FORCE_RETRAIN_BERT = False       # False: reuse ./models/bert if it already exists
USE_SAVED_BERT_OUTPUTS = True    # no torch? reuse data/val_probs_bert.npy + data/test_predictions.csv
RUN_DEMO = False                 # True: launch a small Gradio demo at the end (needs gradio + live model)
BERT_MODEL_NAME = "distilbert-base-uncased"
MAX_LEN, BATCH_SIZE, EPOCHS, LR = 128, 32, 5, 5e-5

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
#  HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def print_header(title, char="=", width=72):
    print(f"\n{char * width}")
    print(f"  {title}")
    print(f"{char * width}")


def force_one_label(scores, pred):
    """Every sentence has >= 1 tactic: if nothing is predicted, take the top-scoring label."""
    pred = pred.copy()
    empty = pred.sum(axis=1) == 0
    pred[empty, scores[empty].argmax(axis=1)] = 1
    return pred


def evaluate_multilabel(name, Y_true, Y_pred, verbose=True):
    """Multilabel metrics: micro/macro F1, exact match, label coverage."""
    tp = (Y_true & Y_pred).sum(axis=0)
    res = {
        "name": name,
        "micro_f1": f1_score(Y_true, Y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(Y_true, Y_pred, average="macro", zero_division=0),
        "exact_match": float((Y_true == Y_pred).all(axis=1).mean()),
        "label_coverage": float((tp > 0).mean()),   # tactics predicted correctly at least once
    }
    if verbose:
        print(f"\n  [{name}]")
        print(f"  Micro-F1:          {res['micro_f1']:.4f}")
        print(f"  Macro-F1:          {res['macro_f1']:.4f}")
        print(f"  Exact match:       {res['exact_match']:.4f}")
        print(f"  Label coverage:    {res['label_coverage']:.4f}")
    return res


def per_tactic_table(Y_true, Y_pred, labels):
    p, r, f, s = precision_recall_fscore_support(Y_true, Y_pred, zero_division=0)
    return pd.DataFrame({"tactic": labels, "precision": p, "recall": r, "f1": f, "support": s})


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

### Step 0: Load the raw STIX data
Reads the ATT&CK Enterprise bundle and prints its version and object types.

In [3]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 0: LOAD RAW DATA
# ═════════════════════════════════════════════════════════════════════════════
print_header("PROJECT: ATT&CK TACTIC CLASSIFICATION (CTI SENTENCES)")

stix_files = sorted(glob.glob(os.path.join(DATA_DIR, "enterprise-attack*.json")))
if not stix_files:
    raise FileNotFoundError(
        "ATT&CK STIX file not found in ./data/.\n"
        "  Download enterprise-attack-20.0.json from:\n"
        "  https://github.com/mitre-attack/attack-stix-data  (folder enterprise-attack/)")

STIX_FILE = stix_files[-1]
print(f"\n  Reading: data/{os.path.basename(STIX_FILE)}")
with open(STIX_FILE, encoding="utf-8") as f:
    bundle = json.load(f)
objects = bundle["objects"]

print(f"  Total STIX objects: {len(objects):,}")
for o in objects:
    if o["type"] == "x-mitre-collection":
        print(f"  Collection: {o.get('name')} | version {o.get('x_mitre_version')}")
        break

print("\n  --- Object types ---")
for t, c in Counter(o["type"] for o in objects).most_common(8):
    print(f"    {t:<28} {c:>8,}")


  PROJECT: ATT&CK TACTIC CLASSIFICATION (CTI SENTENCES)

  Reading: data/enterprise-attack-20_0.json


  Total STIX objects: 26,690
  Collection: Enterprise ATT&CK | version 20.0

  --- Object types ---
    relationship                   21,819
    x-mitre-analytic                1,770
    attack-pattern                    867
    malware                           750
    x-mitre-detection-strategy        705
    course-of-action                  268
    intrusion-set                     193
    x-mitre-data-component            109


### Step 1: Build the dataset
Procedure examples (the `uses` relationships) become sentences. Each technique is mapped to its tactics through `kill_chain_phases`, so every sentence inherits the tactics as labels.

In [4]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 1: BUILD THE (SENTENCE, TECHNIQUE, TACTIC) TABLE
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 1: BUILD THE DATASET FROM ATT&CK PROCEDURE EXAMPLES")

def is_active(o):
    """Skip revoked or deprecated objects."""
    return not o.get("revoked", False) and not o.get("x_mitre_deprecated", False)


def attack_id(o):
    for ref in o.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")
    return None


def strip_markup(t):
    t = re.sub(r"\(Citation:[^)]*\)", "", t)            # (Citation: ...)
    t = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", t)      # [text](url) -> text
    t = re.sub(r"<code>|</code>", "", t)
    return re.sub(r"\s+", " ", t).strip()


# Technique -> tactics (via kill_chain_phases)
techniques = {}
for o in objects:
    if o["type"] == "attack-pattern" and is_active(o):
        tactics = [p["phase_name"] for p in o.get("kill_chain_phases", [])
                   if p.get("kill_chain_name") == "mitre-attack"]
        techniques[o["id"]] = {"technique_id": attack_id(o), "technique_name": o["name"], "tactics": tactics}

# Sources: groups, malware, tools, campaigns
sources = {o["id"]: {"source_name": o["name"], "source_type": o["type"]}
           for o in objects
           if o["type"] in ("intrusion-set", "malware", "tool", "campaign") and is_active(o)}

# Procedure examples = 'uses' relationships that carry a description
rows = []
for o in objects:
    if o["type"] == "relationship" and o.get("relationship_type") == "uses" and is_active(o):
        src, tgt = o["source_ref"], o["target_ref"]
        if src in sources and tgt in techniques and o.get("description"):
            tech, s = techniques[tgt], sources[src]
            rows.append({"text": strip_markup(o["description"]),
                         "technique_id": tech["technique_id"],
                         "technique_name": tech["technique_name"],
                         "tactics": "|".join(tech["tactics"]),
                         "source_id": src,
                         "source_name": s["source_name"],
                         "source_type": s["source_type"]})

df_raw = pd.DataFrame(rows)
print(f"\n  Active techniques / sub-techniques: {len(techniques):,}")
print(f"  Raw procedure examples:             {len(df_raw):,}")

df_raw = df_raw[df_raw["text"].str.split().str.len() >= 5]          # drop very short texts
df_raw = df_raw.drop_duplicates(subset=["text", "technique_id"]).reset_index(drop=True)
print(f"  After filtering:                    {len(df_raw):,}")

print("\n  --- Source types ---")
for t, c in df_raw["source_type"].value_counts().items():
    print(f"    {t:<14} {c:>8,}  ({c / len(df_raw) * 100:5.1f}%)")

print("\n  --- Technique coverage ---")
tc = df_raw["technique_id"].value_counts()
print(f"    Techniques with >= 1 sample:  {len(tc):,}")
print(f"    Techniques with >= 20 samples: {(tc >= 20).sum():,}")
print(f"    Average sentence length:      {df_raw['text'].str.split().str.len().mean():.1f} words")


  STEP 1: BUILD THE DATASET FROM ATT&CK PROCEDURE EXAMPLES



  Active techniques / sub-techniques: 706
  Raw procedure examples:             17,636
  After filtering:                    17,563

  --- Source types ---
    malware          10,672  ( 60.8%)
    intrusion-set     4,810  ( 27.4%)
    campaign          1,213  (  6.9%)
    tool                868  (  4.9%)

  --- Technique coverage ---
    Techniques with >= 1 sample:  619
    Techniques with >= 20 samples: 185


    Average sentence length:      14.6 words


### Step 2: Preprocessing and leakage prevention
Technique IDs are removed, actor names and IOCs are masked, duplicate sentences are merged. Sentences that contain the technique name are flagged for a later check.

In [5]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 2: PREPROCESSING & LEAKAGE PREVENTION
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 2: PREPROCESSING & LEAKAGE PREVENTION")

IOC_PATTERNS = [
    (r"\bhttps?://\S+", " <URL> "),
    (r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " <IP> "),
    (r"\b[a-fA-F0-9]{32,64}\b", " <HASH> "),
    (r"\b(?:HKLM|HKCU|HKEY_[A-Z_]+)\\[^\s,;]+", " <REGKEY> "),
    (r"\b[A-Za-z]:\\[^\s,;]+", " <PATH> "),
    (r"\b[\w.-]+@[\w-]+\.[\w.-]+\b", " <EMAIL> "),
]


def mask_row(row):
    t = row["text"]
    t = re.sub(r"\bT\d{4}(?:\.\d{3})?\b", " ", t)                                   # technique IDs
    t = re.sub(re.escape(str(row["source_name"])), " <ACTOR> ", t, flags=re.I)      # actor name
    for pat, rep in IOC_PATTERNS:                                                   # IOCs
        t = re.sub(pat, rep, t)
    return re.sub(r"\s+", " ", t).strip()


df_raw["text_clean"] = df_raw.apply(mask_row, axis=1)

# Flag (do NOT delete) sentences that contain the technique name verbatim
df_raw["has_tech_name"] = df_raw.apply(
    lambda r: str(r["technique_name"]).lower() in r["text_clean"].lower(), axis=1)
df_raw = df_raw[df_raw["text_clean"].str.split().str.len() >= 5]


def union_join(series, sep="|"):
    return sep.join(sorted({x for s in series for x in str(s).split(sep)}))


# Merge identical sentences coming from several sources: union of labels
df = df_raw.groupby("text_clean", as_index=False).agg(
    tactics=("tactics", union_join),
    technique_ids=("technique_id", lambda s: "|".join(sorted(set(s)))),
    sources=("source_id", lambda s: "|".join(sorted(set(s)))),
    has_tech_name=("has_tech_name", "max"),
)
df["n_sources"] = df["sources"].str.count(r"\|") + 1
df["n_tactics"] = df["tactics"].str.count(r"\|") + 1
df["n_words"] = df["text_clean"].str.split().str.len()

LABELS = sorted({t for ts in df["tactics"].str.split("|") for t in ts})
print(f"\n  Unique sentences:                    {len(df):,}")
print(f"  Number of tactics (labels):          {len(LABELS)}")
print(f"  Sentences with > 1 tactic:           {(df['n_tactics'] > 1).mean() * 100:5.1f}%")
print(f"  Sentences with > 1 technique:        {df['technique_ids'].str.contains(r'[|]').mean() * 100:5.1f}%")
print(f"  Sentences containing technique name: {df['has_tech_name'].mean() * 100:5.1f}%")
print(f"  Sentences from > 1 source:           {(df['n_sources'] > 1).mean() * 100:5.1f}%")

print("\n  --- Samples per tactic (all sentences) ---")
tactic_counts = df["tactics"].str.split("|").explode().value_counts()
for lab, cnt in tactic_counts.items():
    bar = "█" * int(cnt / tactic_counts.max() * 30)
    print(f"    {lab:<24} {cnt:>6,}  {bar}")

print("\n  --- Random examples ---")
for _, r in df.sample(4, random_state=SEED).iterrows():
    print(f"\n    TACTICS: {r['tactics'].replace('|', ', ')}")
    print(textwrap.fill(r["text_clean"][:220], 80, initial_indent="    ", subsequent_indent="    "))


  STEP 2: PREPROCESSING & LEAKAGE PREVENTION



  Unique sentences:                    16,952
  Number of tactics (labels):          15
  Sentences with > 1 tactic:            13.5%
  Sentences with > 1 technique:          0.6%
  Sentences containing technique name:  13.8%
  Sentences from > 1 source:             1.9%

  --- Samples per tactic (all sentences) ---
    stealth                   3,562  ██████████████████████████████
    discovery                 3,337  ████████████████████████████
    execution                 2,334  ███████████████████
    command-and-control       2,183  ██████████████████
    persistence               1,522  ████████████
    privilege-escalation      1,494  ████████████
    collection                1,266  ██████████
    credential-access           887  ███████
    initial-access              660  █████
    defense-impairment          645  █████
    resource-development        611  █████
    exfiltration                399  ███
    impact                      397  ███
    lateral-movement          

### Step 3: Group-aware split
Sentences of one malware family or group are alike, so whole groups go to one split. Rare tactics are placed first so that every tactic appears in train, validation and test.

In [6]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 3: GROUP-AWARE, MULTILABEL-STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 3: GROUP-AWARE SPLIT (NO GROUP IN TWO SETS)")

print("""
  Sentences of the same malware / group are written in a similar style.
  A random split would put near-identical sentences in train AND test (leakage).
  So we split by GROUP (source) and keep the tactic distribution balanced.
""")

# --- 3a. Build groups: sources that share a sentence are merged (union-find) ---
parent = {}


def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


for srcs in df["sources"].str.split("|"):
    find(srcs[0])
    for s in srcs[1:]:
        ra, rb = find(srcs[0]), find(s)
        if ra != rb:
            parent[rb] = ra

df["group_id"] = df["sources"].str.split("|").str[0].map(find)
print(f"  Sources -> groups: {df['sources'].str.split('|').explode().nunique():,} -> {df['group_id'].nunique():,}")

# --- 3b. Multi-hot label matrix ---
mlb = MultiLabelBinarizer(classes=LABELS)
Y_all = mlb.fit_transform(df["tactics"].str.split("|")).astype(int)

# --- 3c. Greedy assignment of groups to splits (rare tactics first) ---
RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
G = {gid: np.append(Y_all[idx].sum(axis=0), len(idx)) for gid, idx in df.groupby("group_id").indices.items()}
total = np.sum(list(G.values()), axis=0).astype(float)
weights = 1.0 / np.maximum(total, 1)
target = {s: total * r for s, r in RATIOS.items()}
current = {s: np.zeros_like(total) for s in RATIOS}


def split_cost(split, vec):
    new = current[split] + vec
    return float((weights * ((new - target[split]) / np.maximum(target[split], 1)) ** 2).sum())


rng = np.random.default_rng(SEED)
order = list(df["group_id"].unique())        # same group order as 03_split_data.py
rng.shuffle(order)
order.sort(key=lambda g: -(G[g][:-1] * weights[:-1]).sum())

assignment = {}
for gid in order:
    best = min(RATIOS, key=lambda s: split_cost(s, G[gid]) - split_cost(s, np.zeros_like(total)))
    assignment[gid] = best
    current[best] += G[gid]

df["split"] = df["group_id"].map(assignment)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
Y_train = mlb.transform(train_df["tactics"].str.split("|")).astype(int)
Y_val = mlb.transform(val_df["tactics"].str.split("|")).astype(int)
Y_test = mlb.transform(test_df["tactics"].str.split("|")).astype(int)

print("\n  --- Split sizes ---")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"    {name:<6} {len(d):>7,} sentences ({len(d) / len(df) * 100:4.1f}%) | {d['group_id'].nunique():>4} groups")

print(f"\n  {'tactic':<24}{'train':>7}{'val':>7}{'test':>7}   test share")
for j, lab in enumerate(LABELS):
    c = [int(Y_train[:, j].sum()), int(Y_val[:, j].sum()), int(Y_test[:, j].sum())]
    print(f"  {lab:<24}{c[0]:>7}{c[1]:>7}{c[2]:>7}   {c[2] / sum(c) * 100:5.1f}%")

# --- 3d. Leakage checks ---
overlap = (set(train_df.group_id) & set(val_df.group_id)) | (set(train_df.group_id) & set(test_df.group_id)) \
          | (set(val_df.group_id) & set(test_df.group_id))
print(f"\n  Group overlap between splits: {len(overlap)} (must be 0)")

vec_chk = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train_df["text_clean"])
X_chk = vec_chk.transform(train_df["text_clean"])
for name, d in [("val", val_df), ("test", test_df)]:
    sim = np.asarray((vec_chk.transform(d["text_clean"]) @ X_chk.T).max(axis=1).todense()).ravel()
    print(f"  {name:<5}: {(sim >= 0.9).mean() * 100:.1f}% of sentences have cosine >= 0.9 to a train sentence "
          f"(median max-similarity {np.median(sim):.2f})")


  STEP 3: GROUP-AWARE SPLIT (NO GROUP IN TWO SETS)

  Sentences of the same malware / group are written in a similar style.
  A random split would put near-identical sentences in train AND test (leakage).
  So we split by GROUP (source) and keep the tactic distribution balanced.



  Sources -> groups: 1,071 -> 690

  --- Split sizes ---
    train   11,752 sentences (69.3%) |  509 groups
    val      2,600 sentences (15.3%) |   91 groups
    test     2,600 sentences (15.3%) |   90 groups

  tactic                    train    val   test   test share
  collection                  881    192    193    15.2%
  command-and-control        1522    331    330    15.1%
  credential-access           590    149    148    16.7%
  defense-impairment          443    101    101    15.7%
  discovery                  2331    505    501    15.0%
  execution                  1641    347    346    14.8%
  exfiltration                278     61     60    15.0%
  impact                      277     60     60    15.1%
  initial-access              445    106    109    16.5%
  lateral-movement            243     67     66    17.6%
  persistence                1045    239    238    15.6%
  privilege-escalation       1021    237    236    15.8%
  reconnaissance              117     32    

  val  : 0.4% of sentences have cosine >= 0.9 to a train sentence (median max-similarity 0.36)


  test : 0.6% of sentences have cosine >= 0.9 to a train sentence (median max-similarity 0.36)


### Step 4: Baseline models
TF-IDF features with one-vs-rest Logistic Regression and Linear SVM. The best baseline is chosen by validation Macro-F1.

In [7]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 4: BASELINE MODELS (TF-IDF + LINEAR CLASSIFIERS, ONE-VS-REST)
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 4: BASELINE MODELS (trained on train, scored on validation)")


def make_vectorizers(use_char):
    parts = [TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)]
    if use_char:
        parts.append(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                     sublinear_tf=True, max_features=150_000))
    return parts


def fit_features(parts, texts):
    return hstack([p.fit_transform(texts) for p in parts]).tocsr()


def transform_features(parts, texts):
    return hstack([p.transform(list(texts)) for p in parts]).tocsr()


baseline_configs = {
    "LR  | word 1-2gram":       (False, lambda: LogisticRegression(C=10, max_iter=2000, class_weight="balanced")),
    "SVM | word 1-2gram":       (False, lambda: LinearSVC(C=0.5, class_weight="balanced")),
    "LR  | word + char":        (True,  lambda: LogisticRegression(C=10, max_iter=2000, class_weight="balanced")),
    "SVM | word + char":        (True,  lambda: LinearSVC(C=0.5, class_weight="balanced")),
}

baseline_results, baseline_store = {}, {}
for name, (use_char, make_clf) in baseline_configs.items():
    t0 = time.time()
    parts = make_vectorizers(use_char)
    X_tr = fit_features(parts, train_df["text_clean"])
    clf = OneVsRestClassifier(make_clf(), n_jobs=-1).fit(X_tr, Y_train)
    scores = clf.decision_function(transform_features(parts, val_df["text_clean"]))
    pred = force_one_label(scores, (scores > 0).astype(int))
    baseline_results[name] = evaluate_multilabel(name, Y_val, pred, verbose=False)
    baseline_results[name]["seconds"] = time.time() - t0
    baseline_store[name] = (parts, clf)

print(f"\n  {'Model':<24}{'Micro-F1':>10}{'Macro-F1':>10}{'Exact':>9}{'Seconds':>9}")
print("  " + "-" * 62)
for name, r in baseline_results.items():
    print(f"  {name:<24}{r['micro_f1']:>10.4f}{r['macro_f1']:>10.4f}{r['exact_match']:>9.4f}{r['seconds']:>9.1f}")

best_base_name = max(baseline_results, key=lambda k: baseline_results[k]["macro_f1"])
svm_parts, svm_clf = baseline_store[best_base_name]
print(f"\n  Best baseline by Macro-F1 (validation): {best_base_name}")


def predict_baseline(texts):
    """Return (pseudo-probabilities, predicted label matrix) of the best baseline."""
    s = svm_clf.decision_function(transform_features(svm_parts, texts))
    return sigmoid(2 * s), force_one_label(s, (s > 0).astype(int))


  STEP 4: BASELINE MODELS (trained on train, scored on validation)



  Model                     Micro-F1  Macro-F1    Exact  Seconds
  --------------------------------------------------------------
  LR  | word 1-2gram          0.7969    0.7530   0.6954      2.0
  SVM | word 1-2gram          0.8043    0.7618   0.7062      1.7
  LR  | word + char           0.8060    0.7615   0.7119      8.8
  SVM | word + char           0.8111    0.7751   0.7200      8.2

  Best baseline by Macro-F1 (validation): SVM | word + char


### Step 5: Improved model (DistilBERT)
Fine-tuned in this session if `torch` is available. Otherwise the saved outputs of an earlier run are reused, so the rest of the notebook still runs.

In [8]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 5: IMPROVED MODEL — FINE-TUNED DISTILBERT
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 5: IMPROVED MODEL — FINE-TUNED DISTILBERT")

# bert_mode: "live"  = the model runs in this session (torch + transformers found)
#            "saved" = reuse saved outputs of an earlier run (data/val_probs_bert.npy, data/test_predictions.csv)
#            None    = transformer not available
bert_mode = None
if RUN_BERT:
    try:
        import torch
        from torch import nn
        from torch.utils.data import DataLoader
        from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                                  get_linear_schedule_with_warmup)
        bert_mode = "live"
    except ImportError:
        print("\n  torch / transformers are NOT visible to this kernel.")
        print(f"  Python used by this kernel: {sys.executable}")
        print("  Fix: run  %pip install torch transformers  in a cell, restart the kernel and run again.")
        print("  (In VS Code also choose the same interpreter with 'Select Kernel'.)")
else:
    print("\n  RUN_BERT = False -> the transformer is not run in this session.")

# --- Fallback: reuse the outputs of an earlier DistilBERT run ---
if bert_mode is None and USE_SAVED_BERT_OUTPUTS:
    f_val = os.path.join(DATA_DIR, "val_probs_bert.npy")
    f_test = os.path.join(DATA_DIR, "test_predictions.csv")
    f_valtxt = os.path.join(DATA_DIR, "val_predictions_bert.csv")
    if os.path.exists(f_val) and os.path.exists(f_test) and os.path.exists(f_valtxt):
        P_saved = np.load(f_val)
        val_txt = pd.read_csv(f_valtxt)["text_clean"].tolist()
        tp = pd.read_csv(f_test)
        v_idx = {t: i for i, t in enumerate(val_txt)}
        t_idx = {t: i for i, t in enumerate(tp["text_clean"])}
        ok = (len(P_saved) == len(val_txt) and all(t in v_idx for t in val_df["text_clean"])
              and all(t in t_idx for t in test_df["text_clean"]))
        if ok:
            P_val_bert = P_saved[[v_idx[t] for t in val_df["text_clean"]]]
            saved_test_pred = mlb.transform(
                tp["pred_bert"].iloc[[t_idx[t] for t in test_df["text_clean"]]].str.split("|")).astype(int)
            bert_mode = "saved"
            print("\n  Using SAVED DistilBERT outputs from an earlier run (matched to this split by sentence text):")
            print("    data/val_probs_bert.npy, data/val_predictions_bert.csv, data/test_predictions.csv")
            print("  The model itself is not loaded, so new text cannot be classified in this session.")
        else:
            print("\n  Saved DistilBERT outputs found but they do NOT match this split -> ignored.")
    else:
        print("\n  No saved DistilBERT outputs found in ./data -> the notebook continues with the baseline only.")
        print("  (needed: val_probs_bert.npy, val_predictions_bert.csv, test_predictions.csv)")

# --- Live mode: load or fine-tune the model ---
if bert_mode == "live":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device: {device} | model: {BERT_MODEL_NAME}")
    saved = os.path.exists(os.path.join(MODEL_DIR, "config.json")) and not FORCE_RETRAIN_BERT
    tok = AutoTokenizer.from_pretrained(MODEL_DIR if saved else BERT_MODEL_NAME)

    def bert_probs(texts, batch_size=64):
        """Tactic probabilities [n, 15] from the current model."""
        texts, out = list(texts), []
        bert_model.eval()
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                enc = tok(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                          padding=True, return_tensors="pt").to(device)
                out.append(torch.sigmoid(bert_model(**enc).logits.float()).cpu().numpy())
        return np.vstack(out)

    if saved:
        print(f"  Loading the saved model from {MODEL_DIR} (set FORCE_RETRAIN_BERT=True to retrain)")
        bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
    else:
        print("  Fine-tuning (the best epoch is selected on the validation set)...")
        torch.manual_seed(SEED)
        bert_model = AutoModelForSequenceClassification.from_pretrained(
            BERT_MODEL_NAME, num_labels=len(LABELS), problem_type="multi_label_classification").to(device)

        enc_train = tok(list(train_df["text_clean"]), truncation=True, max_length=MAX_LEN)["input_ids"]

        def collate(idx):
            batch = tok.pad({"input_ids": [enc_train[i] for i in idx]}, return_tensors="pt")
            batch["labels"] = torch.tensor(Y_train[idx], dtype=torch.float32)
            return batch

        loader = DataLoader(list(range(len(enc_train))), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
        pos = Y_train.sum(axis=0)
        pos_weight = torch.tensor(np.sqrt((len(Y_train) - pos) / np.maximum(pos, 1)),
                                  dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(bert_model.parameters(), lr=LR, weight_decay=0.01)
        total_steps = len(loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

        best_macro = -1.0
        for epoch in range(1, EPOCHS + 1):
            bert_model.train()
            run_loss = 0.0
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                y = batch.pop("labels")
                optimizer.zero_grad()
                with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                    logits = bert_model(**batch).logits
                loss = loss_fn(logits.float(), y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                run_loss += loss.item()
            P_tmp = bert_probs(val_df["text_clean"])
            pred_tmp = force_one_label(P_tmp, (P_tmp >= 0.5).astype(int))
            m = evaluate_multilabel("epoch", Y_val, pred_tmp, verbose=False)
            print(f"    Epoch {epoch}/{EPOCHS} | loss {run_loss / len(loader):.4f} | "
                  f"micro-F1 {m['micro_f1']:.4f} | macro-F1 {m['macro_f1']:.4f} | exact {m['exact_match']:.4f}")
            if m["macro_f1"] > best_macro:           # model selection uses VALIDATION only
                best_macro = m["macro_f1"]
                os.makedirs(MODEL_DIR, exist_ok=True)
                bert_model.save_pretrained(MODEL_DIR)
                tok.save_pretrained(MODEL_DIR)
        bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
        print(f"\n  Best epoch saved to {MODEL_DIR} (validation Macro-F1 {best_macro:.4f})")

    P_val_bert = bert_probs(val_df["text_clean"])

bert_available = bert_mode is not None
if bert_available:
    pred_val_bert = force_one_label(P_val_bert, (P_val_bert >= 0.5).astype(int))
    evaluate_multilabel(f"DistilBERT (validation, threshold 0.5, mode: {bert_mode})", Y_val, pred_val_bert)


  STEP 5: IMPROVED MODEL — FINE-TUNED DISTILBERT

  torch / transformers are NOT visible to this kernel.
  Python used by this kernel: /usr/bin/python3
  Fix: run  %pip install torch transformers  in a cell, restart the kernel and run again.
  (In VS Code also choose the same interpreter with 'Select Kernel'.)

  Using SAVED DistilBERT outputs from an earlier run (matched to this split by sentence text):
    data/val_probs_bert.npy, data/val_predictions_bert.csv, data/test_predictions.csv
  The model itself is not loaded, so new text cannot be classified in this session.

  [DistilBERT (validation, threshold 0.5, mode: saved)]
  Micro-F1:          0.8352
  Macro-F1:          0.8180
  Exact match:       0.7796
  Label coverage:    1.0000


### Step 6: Threshold tuning
Per-tactic thresholds are tuned on one half of the validation set and scored on the other half (grouped folds). They are only used if this honest estimate improves Macro-F1.

In [9]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 6: PER-TACTIC THRESHOLD TUNING (HONEST 2-FOLD GROUPED CV ON VALIDATION)
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 6: PER-TACTIC THRESHOLD TUNING")

GRID = np.round(np.arange(0.10, 0.91, 0.05), 2)


def predict_with_thr(P, thr):
    pred = (P >= thr).astype(int)
    empty = pred.sum(axis=1) == 0
    pred[empty, (P[empty] / thr).argmax(axis=1)] = 1
    return pred


def tune_thresholds(P, Y):
    """Best threshold per tactic = the one maximizing that tactic's F1."""
    thr = np.full(P.shape[1], 0.5)
    for j in range(P.shape[1]):
        sc = [f1_score(Y[:, j], (P[:, j] >= t).astype(int), zero_division=0) for t in GRID]
        thr[j] = GRID[int(np.argmax(sc))]
    return thr


bert_thr = np.full(len(LABELS), 0.5)
if bert_available:
    default_thr = np.full(len(LABELS), 0.5)
    m_def = evaluate_multilabel("default", Y_val, predict_with_thr(P_val_bert, default_thr), verbose=False)
    m_all = evaluate_multilabel("tuned (optimistic)", Y_val,
                                predict_with_thr(P_val_bert, tune_thresholds(P_val_bert, Y_val)), verbose=False)

    pred_cv = np.zeros_like(Y_val)
    for tr_idx, te_idx in GroupKFold(n_splits=2).split(P_val_bert, Y_val, groups=val_df["group_id"]):
        pred_cv[te_idx] = predict_with_thr(P_val_bert[te_idx], tune_thresholds(P_val_bert[tr_idx], Y_val[tr_idx]))
    m_cv = evaluate_multilabel("tuned (2-fold CV)", Y_val, pred_cv, verbose=False)

    print(f"\n  {'Strategy':<36}{'Micro-F1':>10}{'Macro-F1':>10}{'Exact':>9}")
    print("  " + "-" * 65)
    for label, m in [("Default threshold 0.5", m_def), ("Tuned on all of val (optimistic)", m_all),
                     ("Tuned with 2-fold CV (honest)", m_cv)]:
        print(f"  {label:<36}{m['micro_f1']:>10.4f}{m['macro_f1']:>10.4f}{m['exact_match']:>9.4f}")

    use_tuned = m_cv["macro_f1"] > m_def["macro_f1"] + 0.002
    if use_tuned:
        bert_thr = tune_thresholds(P_val_bert, Y_val)
    print(f"\n  Decision: {'USE tuned thresholds' if use_tuned else 'KEEP default 0.5 (tuning does not help in honest CV)'}")
    print("  Tuning on the same data you score on is optimistic: thresholds of rare tactics overfit.")
else:
    print("\n  Skipped (no transformer). The baseline uses the natural decision boundary (score > 0).")

# Final predictors used from here on: name -> function(texts) -> (probabilities, label matrix)
PREDICTORS = {"Baseline": predict_baseline}
if bert_mode == "live":
    def predict_bert(texts):
        P = bert_probs(texts)
        return P, predict_with_thr(P, bert_thr)
    PREDICTORS["DistilBERT"] = predict_bert
elif bert_mode == "saved":
    if not np.allclose(bert_thr, 0.5):
        print("\n  WARNING: saved test predictions were made with threshold 0.5, but tuned thresholds were selected.")
    val_key = {t: i for i, t in enumerate(val_df["text_clean"])}
    test_key = {t: i for i, t in enumerate(test_df["text_clean"])}

    def predict_bert(texts):
        texts = list(texts)
        if all(t in val_key for t in texts):
            P = P_val_bert[[val_key[t] for t in texts]]
            return P, predict_with_thr(P, bert_thr)
        if all(t in test_key for t in texts):
            pred = saved_test_pred[[test_key[t] for t in texts]]
            return pred.astype(float), pred          # test probabilities were not saved: 0/1 labels
        raise RuntimeError("DistilBERT is not loaded (saved outputs only): install torch to classify new text.")
    PREDICTORS["DistilBERT"] = predict_bert
print(f"\n  Predictors available: {list(PREDICTORS)}")


  STEP 6: PER-TACTIC THRESHOLD TUNING



  Strategy                              Micro-F1  Macro-F1    Exact
  -----------------------------------------------------------------
  Default threshold 0.5                   0.8352    0.8180   0.7796
  Tuned on all of val (optimistic)        0.8427    0.8240   0.8065
  Tuned with 2-fold CV (honest)           0.8343    0.8126   0.7900

  Decision: KEEP default 0.5 (tuning does not help in honest CV)
  Tuning on the same data you score on is optimistic: thresholds of rare tactics overfit.

  Predictors available: ['Baseline', 'DistilBERT']


### Step 7: Model comparison on validation
All choices (model, epoch, thresholds) are made here, on validation data only.

In [10]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 7: MODEL COMPARISON ON THE VALIDATION SET
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 7: MODEL COMPARISON (VALIDATION)")

val_out, val_results = {}, {}
for name, fn in PREDICTORS.items():
    P_v, pred_v = fn(val_df["text_clean"])
    val_out[name] = (P_v, pred_v)
    val_results[name] = evaluate_multilabel(name, Y_val, pred_v, verbose=False)

print(f"\n  {'Model':<16}{'Micro-F1':>10}{'Macro-F1':>10}{'Exact':>9}{'Coverage':>10}")
print("  " + "-" * 55)
for name, r in val_results.items():
    print(f"  {name:<16}{r['micro_f1']:>10.4f}{r['macro_f1']:>10.4f}{r['exact_match']:>9.4f}{r['label_coverage']:>10.2f}")

best_name = max(val_results, key=lambda k: val_results[k]["macro_f1"])
print(f"\n  Best model by validation Macro-F1: {best_name}")

if len(PREDICTORS) > 1:
    pb = per_tactic_table(Y_val, val_out["Baseline"][1], LABELS)
    pm = per_tactic_table(Y_val, val_out["DistilBERT"][1], LABELS)
    print(f"\n  {'tactic':<24}{'support':>8}{'Baseline F1':>13}{'BERT F1':>10}{'diff':>8}")
    for i, lab in enumerate(LABELS):
        d = pm["f1"][i] - pb["f1"][i]
        print(f"  {lab:<24}{int(pb['support'][i]):>8}{pb['f1'][i]:>13.3f}{pm['f1'][i]:>10.3f}{d:>+8.3f}")
    print(f"\n  DistilBERT is better on {(pm['f1'] > pb['f1']).sum()} of {len(LABELS)} tactics.")

# Steps 11-12 need a model that can classify NEW text (perturbed sentences, examples)
live_name = best_name if (best_name != "DistilBERT" or bert_mode == "live") else "Baseline"
if live_name != best_name:
    print(f"\n  Note: DistilBERT runs only from saved outputs here, so Steps 11-12 use the '{live_name}' model.")


  STEP 7: MODEL COMPARISON (VALIDATION)



  Model             Micro-F1  Macro-F1    Exact  Coverage
  -------------------------------------------------------
  Baseline            0.8111    0.7751   0.7200      1.00
  DistilBERT          0.8352    0.8180   0.7796      1.00

  Best model by validation Macro-F1: DistilBERT

  tactic                   support  Baseline F1   BERT F1    diff
  collection                   192        0.777     0.822  +0.045
  command-and-control          331        0.861     0.884  +0.023
  credential-access            149        0.822     0.841  +0.019
  defense-impairment           101        0.707     0.732  +0.025
  discovery                    505        0.879     0.917  +0.039
  execution                    347        0.801     0.808  +0.007
  exfiltration                  61        0.773     0.859  +0.087
  impact                        60        0.716     0.878  +0.162
  initial-access               106        0.785     0.824  +0.039
  lateral-movement              67        0.703     0.701

### Step 8: Final evaluation on the test set
Used once. Includes bootstrap confidence intervals and a check on sentences that do not contain the technique name.

In [11]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 8: FINAL EVALUATION ON THE TEST SET (USED ONCE)
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 8: FINAL EVALUATION ON THE TEST SET")

print("""
  All choices (model, epoch, thresholds) were frozen on the validation set.
  The test set is evaluated here ONCE. Do not tune anything after seeing these numbers.
""")

test_out, test_results = {}, {}
for name, fn in PREDICTORS.items():
    P_t, pred_t = fn(test_df["text_clean"])
    test_out[name] = (P_t, pred_t)
    test_results[name] = evaluate_multilabel(name, Y_test, pred_t, verbose=False)

print(f"  {'Model':<16}{'Micro-F1':>10}{'Macro-F1':>10}{'Exact':>9}{'Coverage':>10}")
print("  " + "-" * 55)
for name, r in test_results.items():
    print(f"  {name:<16}{r['micro_f1']:>10.4f}{r['macro_f1']:>10.4f}{r['exact_match']:>9.4f}{r['label_coverage']:>10.2f}")

# --- Bootstrap 95% confidence intervals for Macro-F1 ---
rng = np.random.default_rng(SEED)
N_BOOT = 1000
boot = {name: [] for name in PREDICTORS}
for _ in range(N_BOOT):
    idx = rng.integers(0, len(Y_test), len(Y_test))
    for name in PREDICTORS:
        boot[name].append(f1_score(Y_test[idx], test_out[name][1][idx], average="macro", zero_division=0))

print("\n  --- Bootstrap 95% CI for Macro-F1 (1,000 resamples of the test sentences) ---")
for name in PREDICTORS:
    lo, hi = np.percentile(boot[name], [2.5, 97.5])
    print(f"    {name:<16}[{lo:.3f}, {hi:.3f}]")

if len(PREDICTORS) > 1:
    diff = np.array(boot["DistilBERT"]) - np.array(boot["Baseline"])
    lo, hi = np.percentile(diff, [2.5, 97.5])
    print(f"    Difference (BERT - Baseline): {diff.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]"
          f"  -> {'significant' if lo > 0 else 'NOT clearly significant'}")
    ok_b = (test_out["DistilBERT"][1] == Y_test).all(axis=1)
    ok_s = (test_out["Baseline"][1] == Y_test).all(axis=1)
    print(f"    BERT right & baseline wrong: {(ok_b & ~ok_s).sum()} sentences | "
          f"baseline right & BERT wrong: {(~ok_b & ok_s).sum()} sentences")

# --- Leakage check: only sentences WITHOUT the technique name ---
mask = ~test_df["has_tech_name"].values
print(f"\n  --- Only sentences WITHOUT the technique name (n={mask.sum():,}) ---")
for name in PREDICTORS:
    r = evaluate_multilabel(name, Y_test[mask], test_out[name][1][mask], verbose=False)
    print(f"    {name:<16} micro-F1 {r['micro_f1']:.4f} | macro-F1 {r['macro_f1']:.4f} | exact {r['exact_match']:.4f}")

# --- Per-tactic report of the best model ---
final_name = max(test_results, key=lambda k: test_results[k]["macro_f1"])
print(f"\n  --- Per-tactic results on test: {final_name} ---")
tab = per_tactic_table(Y_test, test_out[final_name][1], LABELS)
print(f"  {'tactic':<24}{'precision':>10}{'recall':>9}{'F1':>8}{'support':>9}")
for _, r in tab.iterrows():
    print(f"  {r['tactic']:<24}{r['precision']:>10.3f}{r['recall']:>9.3f}{r['f1']:>8.3f}{int(r['support']):>9}")


  STEP 8: FINAL EVALUATION ON THE TEST SET

  All choices (model, epoch, thresholds) were frozen on the validation set.
  The test set is evaluated here ONCE. Do not tune anything after seeing these numbers.



  Model             Micro-F1  Macro-F1    Exact  Coverage
  -------------------------------------------------------
  Baseline            0.8054    0.7630   0.7104      1.00
  DistilBERT          0.8352    0.7907   0.7808      1.00



  --- Bootstrap 95% CI for Macro-F1 (1,000 resamples of the test sentences) ---
    Baseline        [0.744, 0.780]
    DistilBERT      [0.772, 0.807]
    Difference (BERT - Baseline): +0.028  95% CI [+0.010, +0.046]  -> significant
    BERT right & baseline wrong: 341 sentences | baseline right & BERT wrong: 158 sentences

  --- Only sentences WITHOUT the technique name (n=2,218) ---
    Baseline         micro-F1 0.7972 | macro-F1 0.7429 | exact 0.7033
    DistilBERT       micro-F1 0.8261 | macro-F1 0.7696 | exact 0.7723

  --- Per-tactic results on test: DistilBERT ---
  tactic                   precision   recall      F1  support
  collection                   0.797    0.772   0.784      193
  command-and-control          0.862    0.888   0.875      330
  credential-access            0.800    0.865   0.831      148
  defense-impairment           0.736    0.772   0.754      101
  discovery                    0.906    0.958   0.931      501
  execution                    0.811    0.87

### Step 9: Error analysis
Where does the model fail: number of tactics, sentence length, confused tactics and confident mistakes.

In [12]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 9: ERROR ANALYSIS (VALIDATION SET, BEST MODEL)
# ═════════════════════════════════════════════════════════════════════════════
print_header(f"STEP 9: ERROR ANALYSIS — {best_name} (validation)")

P_e, pred_e = val_out[best_name]
exact_ok = (Y_val == pred_e).all(axis=1)
print(f"\n  Sentences with at least one wrong label: {(~exact_ok).sum():,} / {len(Y_val):,} "
      f"({(~exact_ok).mean() * 100:.1f}%)")

# --- By number of true tactics ---
print("\n  --- Exact match by number of true tactics ---")
n_true = Y_val.sum(axis=1)
for k in sorted(set(n_true)):
    if (n_true == k).sum() >= 10:
        print(f"    {k} tactic(s): n={(n_true == k).sum():>5}   exact match = {exact_ok[n_true == k].mean():.3f}")

# --- By sentence length ---
print("\n  --- Exact match by sentence length (words) ---")
bins = pd.cut(val_df["n_words"], [0, 10, 20, 40, 10_000], labels=["<=10", "11-20", "21-40", ">40"])
for b in bins.cat.categories:
    m = (bins == b).values
    print(f"    {b:<6} n={m.sum():>5}   exact match = {exact_ok[m].mean():.3f}")

# --- Most common confusions: tactic missed -> tactic predicted instead ---
pairs = Counter()
for yt, yp in zip(Y_val, pred_e):
    for a in np.where((yt == 1) & (yp == 0))[0]:
        for b in np.where((yt == 0) & (yp == 1))[0]:
            pairs[(LABELS[a], LABELS[b])] += 1
print("\n  --- Top confusions (true tactic missed -> wrong tactic predicted) ---")
for (a, b), c in pairs.most_common(8):
    print(f"    {a:<22} -> {b:<22}{c:>4}")

# --- Most confident mistakes ---
err_idx = np.where(~exact_ok)[0]
conf = np.array([max(P_e[i][Y_val[i] == 0].max(initial=0), (1 - P_e[i][Y_val[i] == 1]).max(initial=0)) for i in err_idx])
print("\n  --- Most confident mistakes ---")
for i in err_idx[np.argsort(-conf)[:5]]:
    print(f"\n    TRUE: {', '.join(np.array(LABELS)[Y_val[i] == 1])}")
    print(f"    PRED: {', '.join(np.array(LABELS)[pred_e[i] == 1])}")
    print(textwrap.fill(val_df['text_clean'][i][:230], 80, initial_indent="    ", subsequent_indent="    "))

print("""
  Note: many confident 'errors' are label noise. Labels belong to the TECHNIQUE, not to the
  individual sentence, so a sentence can legitimately look like a different tactic.
""")


  STEP 9: ERROR ANALYSIS — DistilBERT (validation)

  Sentences with at least one wrong label: 573 / 2,600 (22.0%)

  --- Exact match by number of true tactics ---
    1 tactic(s): n= 2255   exact match = 0.799
    2 tactic(s): n=  266   exact match = 0.647
    3 tactic(s): n=   46   exact match = 0.587
    4 tactic(s): n=   33   exact match = 0.788

  --- Exact match by sentence length (words) ---
    <=10   n=  770   exact match = 0.855
    11-20  n= 1384   exact match = 0.764
    21-40  n=  406   exact match = 0.707
    >40    n=   40   exact match = 0.600

  --- Top confusions (true tactic missed -> wrong tactic predicted) ---
    execution              -> stealth                 23
    stealth                -> execution               16
    stealth                -> defense-impairment      14
    persistence            -> stealth                 14
    discovery              -> execution               12
    execution              -> privilege-escalation    11
    stealth       

### Step 10: Security impact analysis
A missed tactic is a blind spot, an extra tactic costs analyst time. Per-tactic recall and false alarms on the test set.

In [13]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 10: SECURITY IMPACT ANALYSIS (TEST SET, BEST MODEL)
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 10: SECURITY IMPACT ANALYSIS")

print("""
  The classifier is decision support for a CTI analyst who maps reports to ATT&CK.
  The cost of errors is asymmetric:
  - False Negative (missed tactic):  a behavior stays unmapped -> a detection blind spot (HIGH COST)
  - False Positive (extra tactic):   the analyst checks and rejects it (MODERATE COST, analyst time)

  Therefore: RECALL per tactic matters most, with precision keeping the workload acceptable.
""")

Y_pred_f = test_out[final_name][1]
tab = per_tactic_table(Y_test, Y_pred_f, LABELS)


def verdict(recall):
    return "OK" if recall >= 0.85 else ("REVIEW" if recall >= 0.75 else "RISK")


print(f"  {'Tactic':<24}{'Recall':>8}{'Precision':>11}{'Missed':>8}{'False alarms':>14}{'Verdict':>10}")
print("  " + "-" * 77)
for j, r in tab.iterrows():
    missed = int(((Y_test[:, j] == 1) & (Y_pred_f[:, j] == 0)).sum())
    false_alarm = int(((Y_test[:, j] == 0) & (Y_pred_f[:, j] == 1)).sum())
    print(f"  {r['tactic']:<24}{r['recall']:>8.3f}{r['precision']:>11.3f}{missed:>8}{false_alarm:>14}{verdict(r['recall']):>10}")

tot_missed = int(((Y_test == 1) & (Y_pred_f == 0)).sum())
tot_fa = int(((Y_test == 0) & (Y_pred_f == 1)).sum())
print(f"\n  Total missed tactic labels: {tot_missed:,} of {int(Y_test.sum()):,} "
      f"({tot_missed / Y_test.sum() * 100:.1f}%)")
print(f"  Total false alarms:         {tot_fa:,} -> about {tot_fa / len(Y_test) * 100:.1f} extra labels per 100 sentences")

weak = tab.sort_values("f1").head(3)
print("\n  Weakest tactics (lowest F1): " + ", ".join(f"{r.tactic} (F1 {r.f1:.2f}, n={int(r.support)})" for r in weak.itertuples()))
print("  Tactics with few test sentences have unstable scores: read them with care.")


  STEP 10: SECURITY IMPACT ANALYSIS

  The classifier is decision support for a CTI analyst who maps reports to ATT&CK.
  The cost of errors is asymmetric:
  - False Negative (missed tactic):  a behavior stays unmapped -> a detection blind spot (HIGH COST)
  - False Positive (extra tactic):   the analyst checks and rejects it (MODERATE COST, analyst time)

  Therefore: RECALL per tactic matters most, with precision keeping the workload acceptable.

  Tactic                    Recall  Precision  Missed  False alarms   Verdict
  -----------------------------------------------------------------------------
  collection                 0.772      0.797      44            38    REVIEW
  command-and-control        0.888      0.862      37            47        OK
  credential-access          0.865      0.800      20            32        OK
  defense-impairment         0.772      0.736      23            28    REVIEW
  discovery                  0.958      0.906      21            50        O

### Step 11: Interpretability
Global view (top words of Logistic Regression) and local view (occlusion: remove one word, measure the drop in probability).

In [14]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 11: INTERPRETABILITY (WORD IMPORTANCE)
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 11: INTERPRETABILITY")

# --- 11a. Top words of a word-level Logistic Regression (global view) ---
vec_w = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
lr_w = OneVsRestClassifier(LogisticRegression(C=10, max_iter=2000, class_weight="balanced"), n_jobs=-1)
lr_w.fit(vec_w.fit_transform(train_df["text_clean"]), Y_train)
names = np.array(vec_w.get_feature_names_out())
print("\n  --- Top words / phrases per tactic (Logistic Regression weights) ---")
for i, lab in enumerate(LABELS):
    top = names[np.argsort(lr_w.estimators_[i].coef_[0])[-6:][::-1]]
    print(f"    {lab:<24}{', '.join(top)}")

# --- 11b. Occlusion: remove one word at a time and measure the drop in probability (local view) ---
def explain_occlusion(sentence, prob_fn):
    words = sentence.split()
    variants = [" ".join(words[:i] + words[i + 1:]) or "." for i in range(len(words))]
    P = np.asarray(prob_fn([sentence] + variants))
    p_full, p_var = P[0], P[1:]
    pred = np.where(p_full >= 0.5)[0]
    pred = pred if len(pred) else np.array([int(p_full.argmax())])
    target = int(pred[np.argmax(p_full[pred])])
    return words, p_full, pred, target, p_full[target] - p_var[:, target]


EXAMPLES = [
    "<ACTOR> used ntdsutil to obtain a copy of the victim environment ntds.dit file.",
    "<ACTOR> has used scheduled tasks to establish persistence for installed tools.",
    "<ACTOR> encrypted files on the network shares and demanded a ransom for the decryption key.",
    "The attackers sent spearphishing emails with a malicious attachment, then used PowerShell to download "
    "a second-stage payload and added a registry run key so it would start at every login.",
]

prob_fn = lambda texts: PREDICTORS[live_name](texts)[0]
print(f"\n  --- Occlusion explanations ({live_name}) ---")
print("  Score = drop in the probability of the explained tactic when the word is removed.")
for s in EXAMPLES:
    words, p_full, pred, target, score = explain_occlusion(s, prob_fn)
    top = np.argsort(-score)[:4]
    print(f"\n    {textwrap.shorten(s, 100)}")
    print(f"    predicted: " + ", ".join(f"{LABELS[i]} ({p_full[i]:.2f})" for i in pred))
    print(f"    key words for '{LABELS[target]}': " + ", ".join(f"{words[i]} ({score[i]:+.2f})" for i in top if score[i] > 0.01))

print("\n  Occlusion looks at one word at a time (misses word interactions) and shows influence, not causation.")


  STEP 11: INTERPRETABILITY



  --- Top words / phrases per tactic (Logistic Regression weights) ---
    collection              keylogging, keystrokes, screenshots, keylogger, data, screen
    command-and-control     c2, traffic, download, proxy, communications, additional
    credential-access       credentials, passwords, keystrokes, keylogging, password, keylogger
    defense-impairment      registry, signed, firewall, disable, logs, modify
    discovery               enumerate, identify, check, discover, checks, list
    execution               shell, powershell, javascript, wmi, execution, python
    exfiltration            exfiltrate, exfiltrated, data, exfiltration, exfiltrates, automatically
    impact                  cryptocurrency, ransomware, wipe, ransom, overwrite, encrypt
    initial-access          credentials, distributed, access, spread, removable, credentials to
    lateral-movement        smb, rdp, laterally, ssh, spread, remote
    persistence             persistence, registry, startup, sched

### Step 12: Robustness
Stability of predictions under synonym replacement, random typos and an appended benign sentence. Analysis only, nothing is retrained.

In [15]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 12: ROBUSTNESS TO MILD INPUT NOISE (ANALYSIS ONLY, MODEL UNCHANGED)
# ═════════════════════════════════════════════════════════════════════════════
print_header(f"STEP 12: ROBUSTNESS — {live_name}")

SYNONYMS = {"used": "leveraged", "uses": "leverages", "use": "leverage", "downloads": "retrieves",
            "download": "retrieve", "executes": "runs", "executed": "ran", "creates": "establishes",
            "created": "established", "obtain": "acquire", "obtained": "acquired", "data": "information",
            "files": "documents", "file": "document", "collect": "gather", "collected": "gathered",
            "sends": "transmits", "sent": "transmitted", "steal": "harvest", "stole": "harvested"}


def synonym_perturb(s, rnd):
    out = []
    for w in s.split():
        core = w.strip(".,;:()")
        rep = SYNONYMS.get(core.lower())
        out.append(w.replace(core, rep) if rep else w)
    return " ".join(out)


def typo_perturb(s, rnd, p=0.15):
    out = []
    for w in s.split():
        if len(w) > 4 and not w.startswith("<") and rnd.random() < p:
            i = rnd.randrange(1, len(w) - 2)
            w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        out.append(w)
    return " ".join(out)


def distractor_perturb(s, rnd):
    return s + " The report was published in 2020."


PERTURBATIONS = {"synonym replacement": synonym_perturb, "random typos": typo_perturb,
                 "benign sentence appended": distractor_perturb}

N_ROB = min(500, len(test_df))
sub_idx = np.random.default_rng(SEED).choice(len(test_df), N_ROB, replace=False)
sub_texts = list(test_df["text_clean"].iloc[sub_idx])
Y_sub = Y_test[sub_idx]
base_pred = PREDICTORS[live_name](sub_texts)[1]

rob_rows = [("original", 1.0, f1_score(Y_sub, base_pred, average="micro", zero_division=0),
             f1_score(Y_sub, base_pred, average="macro", zero_division=0))]
rnd = random.Random(SEED)
for pname, pfn in PERTURBATIONS.items():
    pred_p = PREDICTORS[live_name]([pfn(s, rnd) for s in sub_texts])[1]
    rob_rows.append((pname, float((pred_p == base_pred).all(axis=1).mean()),
                     f1_score(Y_sub, pred_p, average="micro", zero_division=0),
                     f1_score(Y_sub, pred_p, average="macro", zero_division=0)))

print(f"\n  {N_ROB} random test sentences, mild random perturbations:")
print(f"\n  {'Input':<26}{'Labels unchanged':>18}{'Micro-F1':>10}{'Macro-F1':>10}")
print("  " + "-" * 64)
for name, same, mi, ma in rob_rows:
    print(f"  {name:<26}{same * 100:>17.1f}%{mi:>10.3f}{ma:>10.3f}")

print("""
  Compare rows WITHIN this table only (500-sentence subset, not the full test set).
  These are mild random changes, not a targeted adversarial attack.
""")

rob_file = os.path.join(DATA_DIR, "robustness_results.csv")
if os.path.exists(rob_file):
    print("  --- Saved robustness results from an earlier run (data/robustness_results.csv) ---")
    print(pd.read_csv(rob_file).round(3).to_string(index=False))


  STEP 12: ROBUSTNESS — Baseline



  500 random test sentences, mild random perturbations:

  Input                       Labels unchanged  Micro-F1  Macro-F1
  ----------------------------------------------------------------
  original                              100.0%     0.816     0.775
  synonym replacement                    92.2%     0.803     0.764
  random typos                           86.6%     0.777     0.742
  benign sentence appended               84.6%     0.810     0.774

  Compare rows WITHIN this table only (500-sentence subset, not the full test set).
  These are mild random changes, not a targeted adversarial attack.



### Step 13: Visualizations
Four figures are saved as PNG files next to the notebook.

In [16]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 13: VISUALIZATIONS
# ═════════════════════════════════════════════════════════════════════════════
print_header("GENERATING VISUALIZATIONS...")

C_BASE, C_BERT, C_GREY = "#95a5a6", "#2a6fdb", "#c9c6bb"
model_colors = {"Baseline": C_BASE, "DistilBERT": C_BERT}

# --- Figure 1: tactic distribution per split ---
fig1, ax1 = plt.subplots(figsize=(11, 6))
order_lab = list(tactic_counts.index)[::-1]
left = np.zeros(len(order_lab))
for split, col, Yx in [("train", "#34495e", Y_train), ("val", "#e8893a", Y_val), ("test", "#2a6fdb", Y_test)]:
    vals = np.array([Yx[:, LABELS.index(l)].sum() for l in order_lab])
    ax1.barh(order_lab, vals, left=left, label=split, color=col)
    left += vals
ax1.set_xlabel("Number of sentences")
ax1.set_title("Samples per tactic and split (strong class imbalance)", fontweight="bold")
ax1.legend()
ax1.grid(True, alpha=0.3, axis="x")
fig1.tight_layout()
fig1.savefig(os.path.join(SCRIPT_DIR, "fig1_tactic_distribution.png"), dpi=150)
plt.close(fig1)
print("  Saved: fig1_tactic_distribution.png")

# --- Figure 2: model comparison (validation vs test) ---
metrics_show = [("micro_f1", "Micro-F1"), ("macro_f1", "Macro-F1"), ("exact_match", "Exact match")]
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (title, res) in zip(axes2, [("Validation", val_results), ("Test (final)", test_results)]):
    n_models = len(res)
    width = 0.8 / n_models
    for k, (name, r) in enumerate(res.items()):
        xs = np.arange(len(metrics_show)) + (k - (n_models - 1) / 2) * width
        vals = [r[m] for m, _ in metrics_show]
        ax.bar(xs, vals, width, label=name, color=model_colors.get(name, C_GREY))
        for x, v in zip(xs, vals):
            ax.text(x, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
    ax.set_xticks(np.arange(len(metrics_show)))
    ax.set_xticklabels([n for _, n in metrics_show])
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")
    ax.legend(fontsize=9)
axes2[0].set_ylabel("Score")
fig2.suptitle("Model comparison", fontsize=14, fontweight="bold")
fig2.tight_layout()
fig2.savefig(os.path.join(SCRIPT_DIR, "fig2_model_comparison.png"), dpi=150)
plt.close(fig2)
print("  Saved: fig2_model_comparison.png")

# --- Figure 3: per-tactic F1 on test ---
tabs = {n: per_tactic_table(Y_test, test_out[n][1], LABELS) for n in PREDICTORS}
order_idx = np.argsort(tabs[final_name]["f1"].values)
fig3, ax3 = plt.subplots(figsize=(11, 7))
n_models = len(tabs)
h = 0.8 / n_models
for k, (name, t) in enumerate(tabs.items()):
    ys = np.arange(len(LABELS)) + (k - (n_models - 1) / 2) * h
    ax3.barh(ys, t["f1"].values[order_idx], h, label=name, color=model_colors.get(name, C_GREY))
ax3.set_yticks(np.arange(len(LABELS)))
ax3.set_yticklabels([f"{LABELS[i]} (n={int(tabs[final_name]['support'][i])})" for i in order_idx])
ax3.set_xlabel("F1 on the test set")
ax3.set_title("Per-tactic F1 (test)", fontweight="bold")
ax3.set_xlim(0.4, 1.0)
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3, axis="x")
fig3.tight_layout()
fig3.savefig(os.path.join(SCRIPT_DIR, "fig3_per_tactic_f1.png"), dpi=150)
plt.close(fig3)
print("  Saved: fig3_per_tactic_f1.png")

# --- Figure 4: confusion between tactics (missed -> wrongly predicted), test set ---
conf_m = np.zeros((len(LABELS), len(LABELS)), dtype=int)
for yt, yp in zip(Y_test, test_out[final_name][1]):
    for a in np.where((yt == 1) & (yp == 0))[0]:
        for b in np.where((yt == 0) & (yp == 1))[0]:
            conf_m[a, b] += 1
fig4, ax4 = plt.subplots(figsize=(10, 8.5))
im = ax4.imshow(conf_m, cmap="Blues")
ax4.set_xticks(range(len(LABELS)))
ax4.set_yticks(range(len(LABELS)))
ax4.set_xticklabels(LABELS, rotation=75, ha="right", fontsize=8)
ax4.set_yticklabels(LABELS, fontsize=8)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if conf_m[i, j] >= max(3, conf_m.max() * 0.25):
            ax4.text(j, i, conf_m[i, j], ha="center", va="center", fontsize=8,
                     color="white" if conf_m[i, j] > conf_m.max() * 0.6 else "black")
ax4.set_xlabel("Tactic predicted instead")
ax4.set_ylabel("True tactic that was missed")
ax4.set_title(f"Tactic confusion on test ({final_name})", fontweight="bold")
fig4.colorbar(im, ax=ax4, fraction=0.04)
fig4.tight_layout()
fig4.savefig(os.path.join(SCRIPT_DIR, "fig4_tactic_confusion.png"), dpi=150)
plt.close(fig4)
print("  Saved: fig4_tactic_confusion.png")


  GENERATING VISUALIZATIONS...


  Saved: fig1_tactic_distribution.png


  Saved: fig2_model_comparison.png


  Saved: fig3_per_tactic_f1.png


  Saved: fig4_tactic_confusion.png


### Final summary

In [17]:
# ═════════════════════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ═════════════════════════════════════════════════════════════════════════════
print_header("FINAL SUMMARY")

base_t = test_results["Baseline"]
final_t = test_results[final_name]

print(f"""
  Dataset: MITRE ATT&CK Enterprise v20.0 ({len(df):,} unique sentences, {len(LABELS)} tactics)
  Task:    Sentence-level MULTILABEL tactic classification
  Split:   group-aware 70/15/15 ({train_df['group_id'].nunique()} + {val_df['group_id'].nunique()} + {test_df['group_id'].nunique()} groups, no overlap)

  Models tested:
    Baseline:      {best_base_name} (best of 4 TF-IDF configurations)
    Improved:      {('fine-tuned ' + BERT_MODEL_NAME + (' (run live)' if bert_mode == 'live' else ' (saved outputs of an earlier run)')) if bert_available else 'not available in this session'}

  Final test results ({final_name}):
    Micro-F1:    {final_t['micro_f1']:.4f}
    Macro-F1:    {final_t['macro_f1']:.4f}
    Exact match: {final_t['exact_match']:.4f}

  Key Findings:
  - Macro-F1 is the main metric: tactics are very imbalanced (accuracy would hide rare tactics)
  - Group-aware splitting prevents near-duplicate leakage between train and test
  - A contextual model (DistilBERT) usually beats TF-IDF on short behavior sentences
  - Per-tactic threshold tuning must be checked with honest CV; it can overfit rare tactics
  - Errors concentrate in overlapping tactics (Execution / Stealth / Privilege Escalation)
  - Many confident errors are label noise: labels belong to techniques, not sentences
  - Predictions are decision support for analysts and still need human review

  Generated Figures:
  - fig1_tactic_distribution.png   (samples per tactic and split)
  - fig2_model_comparison.png      (validation vs test metrics)
  - fig3_per_tactic_f1.png         (per-tactic F1 on test)
  - fig4_tactic_confusion.png      (which tactics are confused on test)
""")

if final_name != "Baseline":
    print(f"  Baseline test results: micro-F1 {base_t['micro_f1']:.4f} | macro-F1 {base_t['macro_f1']:.4f} "
          f"| exact match {base_t['exact_match']:.4f}")
    print(f"  Macro-F1 gain over the baseline: {final_t['macro_f1'] - base_t['macro_f1']:+.4f}")


  FINAL SUMMARY

  Dataset: MITRE ATT&CK Enterprise v20.0 (16,952 unique sentences, 15 tactics)
  Task:    Sentence-level MULTILABEL tactic classification
  Split:   group-aware 70/15/15 (509 + 91 + 90 groups, no overlap)

  Models tested:
    Baseline:      SVM | word + char (best of 4 TF-IDF configurations)
    Improved:      fine-tuned distilbert-base-uncased (saved outputs of an earlier run)

  Final test results (DistilBERT):
    Micro-F1:    0.8352
    Macro-F1:    0.7907
    Exact match: 0.7808

  Key Findings:
  - Macro-F1 is the main metric: tactics are very imbalanced (accuracy would hide rare tactics)
  - Group-aware splitting prevents near-duplicate leakage between train and test
  - A contextual model (DistilBERT) usually beats TF-IDF on short behavior sentences
  - Per-tactic threshold tuning must be checked with honest CV; it can overfit rare tactics
  - Errors concentrate in overlapping tactics (Execution / Stealth / Privilege Escalation)
  - Many confident errors are 

### Step 14: Optional web demo

In [18]:
# ═════════════════════════════════════════════════════════════════════════════
#  STEP 14: OPTIONAL WEB DEMO (GRADIO)
# ═════════════════════════════════════════════════════════════════════════════
print_header("STEP 14: OPTIONAL WEB DEMO")

if RUN_DEMO and bert_mode == "live":
    import gradio as gr

    def demo_fn(text, sent_no):
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
        if not sents:
            return pd.DataFrame(), []
        P, pred = PREDICTORS["DistilBERT"](sents)
        rows = [{"#": i + 1, "sentence": s,
                 "predicted tactics": ", ".join(f"{LABELS[j]} ({P[i][j]:.2f})" for j in np.where(pred[i] == 1)[0])}
                for i, s in enumerate(sents)]
        k = min(max(int(sent_no or 1), 1), len(sents)) - 1
        words, p_full, pr, target, score = explain_occlusion(sents[k], lambda t: PREDICTORS["DistilBERT"](t)[0])
        return pd.DataFrame(rows), [(w + " ", float(sc)) for w, sc in zip(words, score)]

    gr.Interface(fn=demo_fn,
                 inputs=[gr.Textbox(lines=6, label="Threat report paragraph"),
                         gr.Number(value=1, precision=0, label="Sentence number to explain")],
                 outputs=[gr.Dataframe(label="Predicted ATT&CK tactics"),
                          gr.HighlightedText(label="Word importance (occlusion)")],
                 title="ATT&CK tactic classifier (DistilBERT)").launch(share=True)
else:
    print("\n  Demo skipped. To use it: pip install gradio, set RUN_DEMO = True and run with a live DistilBERT model.")


  STEP 14: OPTIONAL WEB DEMO

  Demo skipped. To use it: pip install gradio, set RUN_DEMO = True and run with a live DistilBERT model.
